In [1]:
import pandas as pd
import numpy as np
import networkx as nx
from itertools import combinations
from pathlib import Path

DATASETS = {
    "BANC": "https://storage.googleapis.com/flywire-data/internship_projects/edge_lists/banc_626_edge_list.csv",
    "FAFB": "https://storage.googleapis.com/flywire-data/internship_projects/edge_lists/fafb_783_edge_list.csv",
    "MANC": "https://storage.googleapis.com/flywire-data/internship_projects/edge_lists/manc_1.2.1_edge_list.csv",
    "MAOL": "https://storage.googleapis.com/flywire-data/internship_projects/edge_lists/maol_1.1_edge_list.csv",
    "MCNS": "https://storage.googleapis.com/flywire-data/internship_projects/edge_lists/mcns_0.9_edge_list.csv",
}

# Create the data directory if it doesn't exist
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

dfs = {}
graphs = {}
node_sets = {}

# ==========================================
# Download, Save, + Load
# ==========================================

for name, url in DATASETS.items():
    print(f"\nProcessing {name}...")

    # Define the local file path
    local_path = data_dir / f"{name.upper()}.csv"

    # Download and save the file locally using pandas
    df = pd.read_csv(url)
    df.to_csv(local_path, index=False)
    print(f"Saved to {local_path}")

    print(name, df.head())
    print("Columns:", list(df.columns))

    dfs[name] = df

print("\nLoaded and saved all datasets.")


Processing BANC...
Saved to data/BANC.csv
BANC      source neuron id    target neuron id
0  720575941350274352  720575941399414563
1  720575941350274352  720575941409918831
2  720575941350274352  720575941425044820
3  720575941350274352  720575941434753568
4  720575941350274352  720575941438392364
Columns: ['source neuron id', 'target neuron id']

Processing FAFB...
Saved to data/FAFB.csv
FAFB      source neuron id    target neuron id
0  720575940596125868  720575940605825666
1  720575940596125868  720575940608552405
2  720575940596125868  720575940609975854
3  720575940596125868  720575940613059993
4  720575940596125868  720575940613599129
Columns: ['source neuron id', 'target neuron id']

Processing MANC...
Saved to data/MANC.csv
MANC    source neuron id  target neuron id
0             10000             10002
1             10000             10016
2             10000             10063
3             10000             10085
4             10000             10102
Columns: ['source neuron

In [ ]:
# CHANGE THESE AFTER INSPECTING THE DATA

SRC = "source neuron id"
DST = "target neuron id"

for name, df in dfs.items():

    G = nx.from_pandas_edgelist(
        df,
        source=SRC,
        target=DST,
        create_using=nx.DiGraph()
    )

    graphs[name] = G
    node_sets[name] = set(G.nodes())

    print(
        f"{name}: "
        f"{G.number_of_nodes():,} nodes, "
        f"{G.number_of_edges():,} edges"
    )

BANC: 112,885 nodes, 2,676,592 edges
FAFB: 138,584 nodes, 3,732,460 edges
MANC: 23,641 nodes, 5,305,638 edges
MAOL: 51,669 nodes, 6,484,936 edges
MCNS: 165,820 nodes, 6,239,112 edges


In [ ]:
print("\nPAIRWISE NODE OVERLAPS")
print("="*50)

for a, b in combinations(node_sets.keys(), 2):

    overlap = len(node_sets[a] & node_sets[b])

    print(
        f"{a:5s} vs {b:5s} "
        f"overlap = {overlap:,}"
    )


PAIRWISE NODE OVERLAPS
BANC  vs FAFB  overlap = 0
BANC  vs MANC  overlap = 0
BANC  vs MAOL  overlap = 0
BANC  vs MCNS  overlap = 0
FAFB  vs MANC  overlap = 0
FAFB  vs MAOL  overlap = 0
FAFB  vs MCNS  overlap = 0
MANC  vs MAOL  overlap = 5,291
MANC  vs MCNS  overlap = 19,314
MAOL  vs MCNS  overlap = 51,614


In [ ]:
from itertools import combinations

print("\nTRIPLE OVERLAPS")
print("="*50)

for trio in combinations(node_sets.keys(), 3):

    common = (
        node_sets[trio[0]]
        & node_sets[trio[1]]
        & node_sets[trio[2]]
    )

    print(trio, len(common))


TRIPLE OVERLAPS
('BANC', 'FAFB', 'MANC') 0
('BANC', 'FAFB', 'MAOL') 0
('BANC', 'FAFB', 'MCNS') 0
('BANC', 'MANC', 'MAOL') 0
('BANC', 'MANC', 'MCNS') 0
('BANC', 'MAOL', 'MCNS') 0
('FAFB', 'MANC', 'MAOL') 0
('FAFB', 'MANC', 'MCNS') 0
('FAFB', 'MAOL', 'MCNS') 0
('MANC', 'MAOL', 'MCNS') 5289


In [ ]:
for name, G in graphs.items():

    indeg = np.array([d for _, d in G.in_degree()])
    outdeg = np.array([d for _, d in G.out_degree()])

    print("\n", name)
    print("-"*30)

    print("nodes:", G.number_of_nodes())
    print("edges:", G.number_of_edges())

    print("mean in-degree :", indeg.mean())
    print("mean out-degree:", outdeg.mean())

    print("max in-degree :", indeg.max())
    print("max out-degree:", outdeg.max())


 BANC
------------------------------
nodes: 112885
edges: 2676592
mean in-degree : 23.71078531248616
mean out-degree: 23.71078531248616
max in-degree : 1723
max out-degree: 1858

 FAFB
------------------------------
nodes: 138584
edges: 3732460
mean in-degree : 26.93283495930266
mean out-degree: 26.93283495930266
max in-degree : 6261
max out-degree: 6523

 MANC
------------------------------
nodes: 23641
edges: 5305638
mean in-degree : 224.4252781185229
mean out-degree: 224.4252781185229
max in-degree : 2486
max out-degree: 4752

 MAOL
------------------------------
nodes: 51669
edges: 6484936
mean in-degree : 125.50922216416033
mean out-degree: 125.50922216416033
max in-degree : 11496
max out-degree: 11215

 MCNS
------------------------------
nodes: 165820
edges: 6239112
mean in-degree : 37.6258111204921
mean out-degree: 37.6258111204921
max in-degree : 6660
max out-degree: 7570


In [ ]:
for name, G in graphs.items():

    comps = sorted(
        nx.weakly_connected_components(G),
        key=len,
        reverse=True
    )

    print(
        f"{name}: "
        f"{len(comps)} components, "
        f"largest={len(comps[0]):,}"
    )

BANC: 8 components, largest=112,871
FAFB: 204 components, largest=136,997
MANC: 1 components, largest=23,641
MAOL: 45 components, largest=51,503
MCNS: 1 components, largest=165,820


In [ ]:
signatures = {}

for name, G in graphs.items():

    rows = []

    for n in G.nodes():

        rows.append({
            "node": n,
            "indeg": G.in_degree(n),
            "outdeg": G.out_degree(n),
        })

    sig = pd.DataFrame(rows)

    signatures[name] = sig

    print(name)
    print(sig.head())

BANC
                 node  indeg  outdeg
0  720575941350274352     65      64
1  720575941399414563    139       6
2  720575941409918831     41      13
3  720575941425044820     23       4
4  720575941434753568    181       6
FAFB
                 node  indeg  outdeg
0  720575940596125868     14      19
1  720575940605825666     34      18
2  720575940608552405    172     115
3  720575940609975854     94      51
4  720575940613059993     84      27
MANC
    node  indeg  outdeg
0  10000    146     201
1  10002    148     186
2  10016   1528    2677
3  10063   1582    2490
4  10085   1303    1011
MAOL
    node  indeg  outdeg
0  10009  11496    9619
1  10015   1119      59
2  10029   1221    6941
3  10046    952   10883
4  10053   1276    7023
MCNS
    node  indeg  outdeg
0  10001    422      64
1  10091    711     249
2  10106    506     315
3  10117    256     211
4  10128    242     191


In [ ]:
for a, b in combinations(signatures.keys(), 2):

    s1 = set(
        zip(
            signatures[a]["indeg"],
            signatures[a]["outdeg"]
        )
    )

    s2 = set(
        zip(
            signatures[b]["indeg"],
            signatures[b]["outdeg"]
        )
    )

    overlap = len(s1 & s2)

    print(
        f"{a} vs {b}: "
        f"{overlap:,} shared degree signatures"
    )

BANC vs FAFB: 7,086 shared degree signatures
BANC vs MANC: 2,739 shared degree signatures
BANC vs MAOL: 6,938 shared degree signatures
BANC vs MCNS: 8,594 shared degree signatures
FAFB vs MANC: 2,693 shared degree signatures
FAFB vs MAOL: 6,990 shared degree signatures
FAFB vs MCNS: 9,125 shared degree signatures
MANC vs MAOL: 5,023 shared degree signatures
MANC vs MCNS: 4,186 shared degree signatures
MAOL vs MCNS: 9,877 shared degree signatures


In [ ]:
datasets = ["BANC", "FAFB", "MANC"]

common = (
    node_sets[datasets[0]]
    & node_sets[datasets[1]]
    & node_sets[datasets[2]]
)

print("Common neurons:", len(common))

subgraphs = {
    d: graphs[d].subgraph(common).copy()
    for d in datasets
}

for d, G in subgraphs.items():
    print(
        d,
        G.number_of_nodes(),
        G.number_of_edges()
    )

Common neurons: 0
BANC 0 0
FAFB 0 0
MANC 0 0


In [ ]:
common = (
    node_sets["MANC"]
    & node_sets["MAOL"]
    & node_sets["MCNS"]
)

len(common)

5289

In [ ]:
common = list(common)

edge_sets = {}

for ds in ["MANC", "MAOL", "MCNS"]:

    G = graphs[ds]

    sub = G.subgraph(common)

    edge_sets[ds] = set(sub.edges())

    print(
        ds,
        len(sub.nodes()),
        len(sub.edges())
    )

print(
    "MANC == MAOL",
    edge_sets["MANC"] == edge_sets["MAOL"]
)

print(
    "MANC == MCNS",
    edge_sets["MANC"] == edge_sets["MCNS"]
)

print(
    "MAOL == MCNS",
    edge_sets["MAOL"] == edge_sets["MCNS"]
)

MANC 5289 179860
MAOL 5289 276933
MCNS 5289 92352
MANC == MAOL False
MANC == MCNS False
MAOL == MCNS False


In [ ]:
for a,b in [
    ("MANC","MCNS"),
    ("MAOL","MCNS"),
    ("MANC","MAOL")
]:

    inter = len(edge_sets[a] & edge_sets[b])

    union = len(edge_sets[a] | edge_sets[b])

    print(
        a,b,
        "Jaccard =", inter/union
    )

MANC MCNS Jaccard = 0.005188196760054209
MAOL MCNS Jaccard = 0.2897001403954822
MANC MAOL Jaccard = 0.008128269073734855


In [ ]:
from collections import Counter
import pandas as pd

degree_signature_counts = {}

for name, G in graphs.items():

    sigs = [
        (G.in_degree(n), G.out_degree(n))
        for n in G.nodes()
    ]

    cnt = Counter(sigs)

    degree_signature_counts[name] = cnt

    print(f"\n{name}")
    print("=" * 50)

    print("Unique degree signatures:",
          len(cnt))

    print("\nMost common signatures:")

    for sig, freq in cnt.most_common(20):
        print(
            f"{sig} -> {freq}"
        )


BANC
Unique degree signatures: 13061

Most common signatures:
(1, 0) -> 1077
(1, 1) -> 774
(2, 0) -> 722
(2, 1) -> 665
(1, 2) -> 661
(2, 2) -> 648
(2, 3) -> 609
(0, 1) -> 600
(3, 0) -> 591
(3, 1) -> 589
(3, 2) -> 588
(2, 4) -> 555
(4, 1) -> 545
(1, 3) -> 544
(3, 4) -> 536
(3, 3) -> 526
(4, 3) -> 517
(4, 2) -> 494
(4, 0) -> 494
(3, 5) -> 493

FAFB
Unique degree signatures: 14056

Most common signatures:
(0, 3) -> 2134
(0, 4) -> 1893
(0, 5) -> 1215
(0, 2) -> 1128
(1, 4) -> 771
(1, 5) -> 667
(1, 3) -> 486
(0, 6) -> 478
(0, 1) -> 452
(1, 6) -> 382
(9, 10) -> 364
(7, 1) -> 356
(10, 10) -> 347
(8, 9) -> 343
(9, 9) -> 339
(9, 8) -> 339
(10, 9) -> 337
(8, 10) -> 336
(12, 10) -> 333
(10, 12) -> 331

MANC
Unique degree signatures: 20675

Most common signatures:
(20, 75) -> 7
(2, 0) -> 7
(25, 87) -> 6
(38, 108) -> 6
(47, 152) -> 6
(36, 100) -> 6
(37, 104) -> 6
(34, 96) -> 6
(30, 103) -> 6
(17, 71) -> 6
(24, 108) -> 5
(33, 102) -> 5
(24, 73) -> 5
(36, 108) -> 5
(10, 80) -> 5
(42, 126) -> 5
(23, 7

In [ ]:
signature_sets = {
    ds: set(cnt.keys())
    for ds, cnt in degree_signature_counts.items()
}

shared_all = (
    signature_sets["BANC"]
    & signature_sets["FAFB"]
    & signature_sets["MANC"]
    & signature_sets["MAOL"]
    & signature_sets["MCNS"]
)

print(
    "Degree signatures shared by ALL datasets:",
    len(shared_all)
)

Degree signatures shared by ALL datasets: 1013


In [ ]:
unique_sigs = {}

for ds, cnt in degree_signature_counts.items():

    unique_sigs[ds] = {
        sig
        for sig, freq in cnt.items()
        if freq == 1
    }

    print(
        ds,
        len(unique_sigs[ds]),
        "unique signatures"
    )

BANC 7314 unique signatures
FAFB 7517 unique signatures
MANC 18372 unique signatures
MAOL 16571 unique signatures
MCNS 11596 unique signatures


In [ ]:
from itertools import combinations

for trio in combinations(unique_sigs.keys(), 3):

    common = (
        unique_sigs[trio[0]]
        & unique_sigs[trio[1]]
        & unique_sigs[trio[2]]
    )

    print(
        trio,
        len(common)
    )

('BANC', 'FAFB', 'MANC') 230
('BANC', 'FAFB', 'MAOL') 263
('BANC', 'FAFB', 'MCNS') 303
('BANC', 'MANC', 'MAOL') 282
('BANC', 'MANC', 'MCNS') 260
('BANC', 'MAOL', 'MCNS') 342
('FAFB', 'MANC', 'MAOL') 248
('FAFB', 'MANC', 'MCNS') 237
('FAFB', 'MAOL', 'MCNS') 346
('MANC', 'MAOL', 'MCNS') 355


In [ ]:
import pandas as pd
import networkx as nx
import numpy as np

from collections import Counter
from pathlib import Path
from datetime import datetime
import time

# ============================================================
# CONFIG
# ============================================================

OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(exist_ok=True)

LOGFILE = OUTPUT_DIR / "wl_run.log"

# ============================================================
# LOGGING
# ============================================================

with open(LOGFILE, "w") as f:
    f.write("WL FEATURE ANALYSIS\n")


def log(msg):

    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    line = f"[{ts}] {msg}"

    print(line)

    with open(LOGFILE, "a") as f:
        f.write(line + "\n")


# ============================================================
# WL IMPLEMENTATION
# ============================================================

def initialize_degree_labels(G):

    return {
        n: (
            G.in_degree(n),
            G.out_degree(n)
        )
        for n in G.nodes()
    }


def relabel_to_integers(labels):

    mapping = {}

    out = {}

    next_id = 0

    for node, label in labels.items():

        if label not in mapping:

            mapping[label] = next_id

            next_id += 1

        out[node] = mapping[label]

    return out


def wl_round(G, labels):

    raw = {}

    for node in G.nodes():

        in_labels = tuple(
            sorted(
                labels[n]
                for n in G.predecessors(node)
            )
        )

        out_labels = tuple(
            sorted(
                labels[n]
                for n in G.successors(node)
            )
        )

        raw[node] = (
            labels[node],
            in_labels,
            out_labels
        )

    return relabel_to_integers(raw)


# ============================================================
# NEIGHBORHOOD FEATURES
# ============================================================

def safe_mean(values):

    if len(values) == 0:
        return 0.0

    return float(np.mean(values))


def neighborhood_features(G):

    indeg = dict(G.in_degree())
    outdeg = dict(G.out_degree())

    feats = {}

    for node in G.nodes():

        preds = list(G.predecessors(node))
        succs = list(G.successors(node))

        feats[node] = {

            "mean_pred_indeg":
                safe_mean(
                    [indeg[p] for p in preds]
                ),

            "mean_pred_outdeg":
                safe_mean(
                    [outdeg[p] for p in preds]
                ),

            "mean_succ_indeg":
                safe_mean(
                    [indeg[s] for s in succs]
                ),

            "mean_succ_outdeg":
                safe_mean(
                    [outdeg[s] for s in succs]
                ),
        }

    return feats


# ============================================================
# MAIN
# ============================================================

summary_rows = []

for ds_name, G in graphs.items():

    start = time.time()

    log("=" * 70)
    log(f"PROCESSING {ds_name}")

    n_nodes = G.number_of_nodes()
    n_edges = G.number_of_edges()

    log(f"Nodes: {n_nodes:,}")
    log(f"Edges: {n_edges:,}")

    # --------------------------------------------------------
    # DEGREE LABELS
    # --------------------------------------------------------

    degree_labels = initialize_degree_labels(G)

    degree_counter = Counter(
        degree_labels.values()
    )

    degree_singletons = sum(
        v == 1
        for v in degree_counter.values()
    )

    log(
        f"Degree classes: "
        f"{len(degree_counter):,}"
    )

    log(
        f"Degree singletons: "
        f"{degree_singletons:,}"
    )

    # --------------------------------------------------------
    # WL-1
    # --------------------------------------------------------

    wl0 = relabel_to_integers(
        degree_labels
    )

    wl1 = wl_round(G, wl0)

    wl1_counter = Counter(
        wl1.values()
    )

    wl1_singletons = sum(
        v == 1
        for v in wl1_counter.values()
    )

    log(
        f"WL1 classes: "
        f"{len(wl1_counter):,}"
    )

    log(
        f"WL1 singletons: "
        f"{wl1_singletons:,}"
    )

    # --------------------------------------------------------
    # WL-2
    # --------------------------------------------------------

    wl2 = wl_round(G, wl1)

    wl2_counter = Counter(
        wl2.values()
    )

    wl2_singletons = sum(
        v == 1
        for v in wl2_counter.values()
    )

    log(
        f"WL2 classes: "
        f"{len(wl2_counter):,}"
    )

    log(
        f"WL2 singletons: "
        f"{wl2_singletons:,}"
    )

    # --------------------------------------------------------
    # NEIGHBOR FEATURES
    # --------------------------------------------------------

    log("Computing neighborhood features...")

    neigh_feats = neighborhood_features(G)

    # --------------------------------------------------------
    # BUILD TABLE
    # --------------------------------------------------------

    rows = []

    for node in G.nodes():

        indeg = G.in_degree(node)
        outdeg = G.out_degree(node)

        rows.append({

            "node": node,

            "indegree": indeg,
            "outdegree": outdeg,

            "total_degree":
                indeg + outdeg,

            "degree_ratio":
                outdeg / max(indeg, 1),

            "mean_pred_indeg":
                neigh_feats[node]["mean_pred_indeg"],

            "mean_pred_outdeg":
                neigh_feats[node]["mean_pred_outdeg"],

            "mean_succ_indeg":
                neigh_feats[node]["mean_succ_indeg"],

            "mean_succ_outdeg":
                neigh_feats[node]["mean_succ_outdeg"],

            "wl1":
                wl1[node],

            "wl1_freq":
                wl1_counter[
                    wl1[node]
                ],

            "wl2":
                wl2[node],

            "wl2_freq":
                wl2_counter[
                    wl2[node]
                ],
        })

    feature_df = pd.DataFrame(rows)

    out_file = (
        OUTPUT_DIR /
        f"{ds_name}_features.csv"
    )

    feature_df.to_csv(
        out_file,
        index=False
    )

    log(
        f"Saved {out_file}"
    )

    runtime = time.time() - start

    summary_rows.append({

        "dataset": ds_name,

        "nodes": n_nodes,
        "edges": n_edges,

        "degree_classes":
            len(degree_counter),

        "degree_singletons":
            degree_singletons,

        "wl1_classes":
            len(wl1_counter),

        "wl1_singletons":
            wl1_singletons,

        "wl2_classes":
            len(wl2_counter),

        "wl2_singletons":
            wl2_singletons,

        "runtime_seconds":
            runtime,
    })

    log(
        f"Runtime: "
        f"{runtime:.2f} sec"
    )

# ============================================================
# SAVE SUMMARY
# ============================================================

summary_df = pd.DataFrame(
    summary_rows
)

summary_df.to_csv(
    OUTPUT_DIR / "wl_summary.csv",
    index=False
)

log("Saved wl_summary.csv")
log("DONE")

[2026-06-03 13:49:19] ======================================================================
[2026-06-03 13:49:19] PROCESSING BANC
[2026-06-03 13:49:19] Nodes: 112,885
[2026-06-03 13:49:19] Edges: 2,676,592
[2026-06-03 13:49:19] Degree classes: 13,061
[2026-06-03 13:49:19] Degree singletons: 7,314
[2026-06-03 13:49:24] WL1 classes: 112,107
[2026-06-03 13:49:24] WL1 singletons: 111,762
[2026-06-03 13:49:28] WL2 classes: 112,574
[2026-06-03 13:49:28] WL2 singletons: 112,411
[2026-06-03 13:49:28] Computing neighborhood features...
[2026-06-03 13:49:39] Saved results/BANC_features.csv
[2026-06-03 13:49:39] Runtime: 20.31 sec
[2026-06-03 13:49:39] ======================================================================
[2026-06-03 13:49:39] PROCESSING FAFB
[2026-06-03 13:49:39] Nodes: 138,584
[2026-06-03 13:49:39] Edges: 3,732,460
[2026-06-03 13:49:40] Degree classes: 14,056
[2026-06-03 13:49:40] Degree singletons: 7,517
[2026-06-03 13:49:44] WL1 classes: 135,208
[2026-06-03 13:49:44] WL1 sin

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx

import hashlib
import json
import math
import time

from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations
from datetime import datetime

# =====================================================
# CONFIG
# =====================================================

OUTDIR = Path("results")
OUTDIR.mkdir(exist_ok=True)

LOGFILE = OUTDIR / "run.log"

# =====================================================
# LOGGING
# =====================================================

def log(msg):

    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    line = f"[{ts}] {msg}"

    print(line)

    with open(LOGFILE, "a") as f:
        f.write(line + "\n")

# =====================================================
# HASHING
# =====================================================

def stable_hash(obj):

    return hashlib.blake2b(
        repr(obj).encode(),
        digest_size=16
    ).hexdigest()

# =====================================================
# ENTROPY
# =====================================================

def entropy(counter):

    total = sum(counter.values())

    p = np.array(
        [v/total for v in counter.values()]
    )

    return float(
        -(p * np.log2(p)).sum()
    )

# =====================================================
# WL
# =====================================================

def wl0_labels(G):

    return {

        n: stable_hash(
            (
                G.in_degree(n),
                G.out_degree(n)
            )
        )

        for n in G.nodes()
    }

def wl_iteration(G, labels):

    out = {}

    for node in G.nodes():

        preds = sorted(
            labels[x]
            for x in G.predecessors(node)
        )

        succs = sorted(
            labels[x]
            for x in G.successors(node)
        )

        signature = (
            labels[node],
            tuple(preds),
            tuple(succs)
        )

        out[node] = stable_hash(signature)

    return out

# =====================================================
# NEIGHBOR FEATURES
# =====================================================

def safe_mean(x):

    if len(x) == 0:
        return 0.0

    return float(np.mean(x))

def neighborhood_stats(G):

    indeg = dict(G.in_degree())
    outdeg = dict(G.out_degree())

    feats = {}

    for node in G.nodes():

        preds = list(G.predecessors(node))
        succs = list(G.successors(node))

        feats[node] = {

            "mean_pred_indeg":
                safe_mean(
                    [indeg[p] for p in preds]
                ),

            "mean_pred_outdeg":
                safe_mean(
                    [outdeg[p] for p in preds]
                ),

            "mean_succ_indeg":
                safe_mean(
                    [indeg[s] for s in succs]
                ),

            "mean_succ_outdeg":
                safe_mean(
                    [outdeg[s] for s in succs]
                )
        }

    return feats

# =====================================================
# STORAGE
# =====================================================

dataset_stats = {}

feature_tables = {}

wl1_hash_sets = {}
wl2_hash_sets = {}

shared_hash_inventory = {
    "WL1": defaultdict(dict),
    "WL2": defaultdict(dict)
}

# =====================================================
# MAIN
# =====================================================

for ds, G in graphs.items():

    log("="*70)
    log(f"PROCESSING {ds}")

    t0 = time.time()

    n_nodes = G.number_of_nodes()
    n_edges = G.number_of_edges()

    log(f"Nodes: {n_nodes:,}")
    log(f"Edges: {n_edges:,}")

    # ---------------------------------------
    # WL
    # ---------------------------------------

    wl0 = wl0_labels(G)

    wl1 = wl_iteration(G, wl0)

    wl2 = wl_iteration(G, wl1)

    wl1_counts = Counter(
        wl1.values()
    )

    wl2_counts = Counter(
        wl2.values()
    )

    wl1_hash_sets[ds] = set(
        wl1_counts.keys()
    )

    wl2_hash_sets[ds] = set(
        wl2_counts.keys()
    )

    # ---------------------------------------
    # Inventory
    # ---------------------------------------

    for node, h in wl1.items():

        shared_hash_inventory["WL1"][h].setdefault(
            ds,
            []
        ).append(node)

    for node, h in wl2.items():

        shared_hash_inventory["WL2"][h].setdefault(
            ds,
            []
        ).append(node)

    # ---------------------------------------
    # Neighborhood features
    # ---------------------------------------

    log("Computing neighborhood stats")

    neigh = neighborhood_stats(G)

    rows = []

    for node in G.nodes():

        indeg = G.in_degree(node)
        outdeg = G.out_degree(node)

        rows.append({

            "node": node,

            "indegree": indeg,
            "outdegree": outdeg,
            "total_degree": indeg + outdeg,

            "mean_pred_indeg":
                neigh[node]["mean_pred_indeg"],

            "mean_pred_outdeg":
                neigh[node]["mean_pred_outdeg"],

            "mean_succ_indeg":
                neigh[node]["mean_succ_indeg"],

            "mean_succ_outdeg":
                neigh[node]["mean_succ_outdeg"],

            "wl1_hash": wl1[node],
            "wl1_freq": wl1_counts[wl1[node]],

            "wl2_hash": wl2[node],
            "wl2_freq": wl2_counts[wl2[node]]
        })

    df = pd.DataFrame(rows)

    feature_tables[ds] = df

    df.to_csv(
        OUTDIR / f"{ds}_features.csv",
        index=False
    )

    runtime = time.time() - t0

    dataset_stats[ds] = {

        "nodes": n_nodes,
        "edges": n_edges,

        "wl1_classes":
            len(wl1_counts),

        "wl2_classes":
            len(wl2_counts),

        "wl1_singletons":
            int(
                sum(
                    v == 1
                    for v in wl1_counts.values()
                )
            ),

        "wl2_singletons":
            int(
                sum(
                    v == 1
                    for v in wl2_counts.values()
                )
            ),

        "wl1_entropy":
            entropy(wl1_counts),

        "wl2_entropy":
            entropy(wl2_counts),

        "wl1_collision_rate":
            1 -
            len(wl1_counts)/n_nodes,

        "wl2_collision_rate":
            1 -
            len(wl2_counts)/n_nodes,

        "runtime_seconds":
            runtime
    }

    log(
        f"WL1 classes: {len(wl1_counts):,}"
    )

    log(
        f"WL2 classes: {len(wl2_counts):,}"
    )

    log(
        f"Runtime: {runtime:.2f}s"
    )

# =====================================================
# SAVE DATASET STATS
# =====================================================

with open(
    OUTDIR / "dataset_statistics.json",
    "w"
) as f:

    json.dump(
        dataset_stats,
        f,
        indent=2
    )

# =====================================================
# PAIRWISE OVERLAPS
# =====================================================

pairwise = {}

for a, b in combinations(graphs.keys(), 2):

    pairwise[f"{a}_{b}"] = {

        "shared_wl1":

            len(
                wl1_hash_sets[a]
                &
                wl1_hash_sets[b]
            ),

        "shared_wl2":

            len(
                wl2_hash_sets[a]
                &
                wl2_hash_sets[b]
            )
    }

with open(
    OUTDIR / "pairwise_overlap.json",
    "w"
) as f:

    json.dump(
        pairwise,
        f,
        indent=2
    )

# =====================================================
# TRIPLE OVERLAPS
# =====================================================

triples = {}

for trio in combinations(
    graphs.keys(),
    3
):

    triples["_".join(trio)] = {

        "shared_wl1":

            len(
                wl1_hash_sets[trio[0]]
                &
                wl1_hash_sets[trio[1]]
                &
                wl1_hash_sets[trio[2]]
            ),

        "shared_wl2":

            len(
                wl2_hash_sets[trio[0]]
                &
                wl2_hash_sets[trio[1]]
                &
                wl2_hash_sets[trio[2]]
            )
    }

with open(
    OUTDIR / "triple_overlap.json",
    "w"
) as f:

    json.dump(
        triples,
        f,
        indent=2
    )

# =====================================================
# SHARED HASH INVENTORY
# =====================================================

shared_export = {
    "WL1": {},
    "WL2": {}
}

for level in ["WL1", "WL2"]:

    for h, datasets in shared_hash_inventory[level].items():

        if len(datasets) >= 2:

            shared_export[level][h] = {

                ds: len(nodes)

                for ds, nodes
                in datasets.items()
            }

with open(
    OUTDIR / "shared_wl_hashes.json",
    "w"
) as f:

    json.dump(
        shared_export,
        f,
        indent=2
    )

# =====================================================
# FINAL REPORT
# =====================================================

print()
print("="*70)
print("FINAL REPORT")
print("="*70)

best_pair = max(
    pairwise.items(),
    key=lambda x:
        x[1]["shared_wl2"]
)

best_triple = max(
    triples.items(),
    key=lambda x:
        x[1]["shared_wl2"]
)

print()
print("BEST PAIR")
print(best_pair)

print()
print("BEST TRIPLE")
print(best_triple)

print()
print("Outputs written to:")
print(OUTDIR)

[2026-06-03 14:08:44] ======================================================================
[2026-06-03 14:08:44] PROCESSING BANC
[2026-06-03 14:08:44] Nodes: 112,885
[2026-06-03 14:08:44] Edges: 2,676,592
[2026-06-03 14:09:17] Computing neighborhood stats
[2026-06-03 14:09:29] WL1 classes: 112,107
[2026-06-03 14:09:29] WL2 classes: 112,574
[2026-06-03 14:09:29] Runtime: 44.95s
[2026-06-03 14:09:29] ======================================================================
[2026-06-03 14:09:29] PROCESSING FAFB
[2026-06-03 14:09:29] Nodes: 138,584
[2026-06-03 14:09:29] Edges: 3,732,460
[2026-06-03 14:09:53] Computing neighborhood stats
[2026-06-03 14:10:08] WL1 classes: 135,208
[2026-06-03 14:10:08] WL2 classes: 135,313
[2026-06-03 14:10:08] Runtime: 38.84s
[2026-06-03 14:10:08] ======================================================================
[2026-06-03 14:10:08] PROCESSING MANC
[2026-06-03 14:10:08] Nodes: 23,641
[2026-06-03 14:10:08] Edges: 5,305,638
[2026-06-03 14:10:30] Computin

In [ ]:
%pip install umap-learn

In [ ]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import HDBSCAN

RESULTS_DIR = "results"

datasets = [
    "BANC",
    "FAFB",
    "MANC",
    "MAOL",
    "MCNS"
]

all_rows = []

for ds in datasets:

    path = os.path.join(
        RESULTS_DIR,
        f"{ds}_features.csv"
    )

    df = pd.read_csv(path)

    df["dataset"] = ds

    all_rows.append(df)

master = pd.concat(
    all_rows,
    ignore_index=True
)

print(master.shape)

# --------------------------------------------------
# feature engineering
# --------------------------------------------------

master["log_indegree"] = np.log1p(
    master["indegree"]
)

master["log_outdegree"] = np.log1p(
    master["outdegree"]
)

master["log_total_degree"] = np.log1p(
    master["total_degree"]
)

feature_cols = [

    "indegree",
    "outdegree",
    "total_degree",

    "mean_pred_indeg",
    "mean_pred_outdeg",

    "mean_succ_indeg",
    "mean_succ_outdeg",

    "log_indegree",
    "log_outdegree",
    "log_total_degree",
]

X = master[feature_cols].fillna(0)

# --------------------------------------------------
# scale
# --------------------------------------------------

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# --------------------------------------------------
# PCA
# --------------------------------------------------

pca = PCA(
    n_components=0.95,   # keep 95% variance
    random_state=42
)

X_pca = pca.fit_transform(X_scaled)

print("PCA dimensions:", X_pca.shape[1])
print(
    "Explained variance:",
    pca.explained_variance_ratio_.sum()
)

# --------------------------------------------------
# HDBSCAN
# --------------------------------------------------

clusterer = HDBSCAN(
    min_cluster_size=50,
    min_samples=20
)

labels = clusterer.fit_predict(X_pca)

master["cluster"] = labels

print(
    "Clusters:",
    len(set(labels)) - (1 if -1 in labels else 0)
)

print(
    "Noise:",
    (labels == -1).mean()
)

# --------------------------------------------------
# cluster composition
# --------------------------------------------------

cluster_stats = {}

for cluster_id in sorted(set(labels)):

    if cluster_id == -1:
        continue

    subset = master[
        master.cluster == cluster_id
    ]

    counts = (
        subset["dataset"]
        .value_counts()
        .to_dict()
    )

    cluster_stats[
        int(cluster_id)
    ] = counts

with open(
    "cluster_composition.json",
    "w"
) as f:

    json.dump(
        cluster_stats,
        f,
        indent=2
    )

# --------------------------------------------------
# save embedding
# --------------------------------------------------

master[
    [
        "node",
        "dataset",
        "cluster"
    ]
].to_csv(
    "neuron_clusters.csv",
    index=False
)

# --------------------------------------------------
# summary
# --------------------------------------------------

print("\nTOP MIXED CLUSTERS\n")

for cid, counts in sorted(
    cluster_stats.items(),
    key=lambda x: len(x[1]),
    reverse=True
)[:20]:

    print(
        f"Cluster {cid}"
    )

    print(counts)

    print()

(492599, 13)
PCA dimensions: 5
Explained variance: 0.9722240820071395
Clusters: 362
Noise: 0.8124417629755643

TOP MIXED CLUSTERS

Cluster 21
{'BANC': 576, 'MAOL': 130, 'MCNS': 48, 'FAFB': 40, 'MANC': 2}

Cluster 31
{'BANC': 71, 'MANC': 2, 'FAFB': 1, 'MAOL': 1, 'MCNS': 1}

Cluster 34
{'BANC': 208, 'MAOL': 6, 'FAFB': 5, 'MCNS': 3, 'MANC': 1}

Cluster 116
{'BANC': 578, 'FAFB': 99, 'MCNS': 60, 'MAOL': 18, 'MANC': 4}

Cluster 124
{'BANC': 451, 'MCNS': 238, 'FAFB': 180, 'MAOL': 11, 'MANC': 3}

Cluster 217
{'MCNS': 3379, 'FAFB': 2581, 'MAOL': 9, 'BANC': 7, 'MANC': 1}

Cluster 13
{'MCNS': 50, 'BANC': 29, 'FAFB': 7, 'MAOL': 2}

Cluster 15
{'BANC': 85, 'MCNS': 27, 'FAFB': 9, 'MAOL': 3}

Cluster 18
{'BANC': 61, 'MCNS': 8, 'MAOL': 5, 'FAFB': 4}

Cluster 20
{'BANC': 34, 'MCNS': 17, 'FAFB': 4, 'MAOL': 1}

Cluster 22
{'BANC': 50, 'FAFB': 1, 'MANC': 1, 'MAOL': 1}

Cluster 23
{'BANC': 75, 'MCNS': 15, 'FAFB': 5, 'MAOL': 1}

Cluster 24
{'BANC': 368, 'MCNS': 136, 'FAFB': 47, 'MAOL': 29}

Cluster 26
{'MCN

In [ ]:
"""
FlyWire Qualification Challenge — Circuit Discovery Script
==========================================================

Strategy
--------
1. Download edge lists for all 5 datasets (already done in EDA, re-done here
   for self-containment).
2. Download cell-type metadata from Codex for FAFB and BANC (the only two
   datasets that expose bulk metadata via the Codex download API without login).
3. For MANC / MAOL / MCNS, fetch metadata from the Janelia neuPrint / DVID
   REST endpoints, which are publicly accessible without authentication.
4. Find cell types that appear in at least 3 datasets → these are our
   candidate neuron sets (biologically proposed homologs).
5. For each candidate cell type group, extract the induced subgraph from each
   dataset's edge list and check for exact directed isomorphism.
6. Grow circuits greedily: for each verified isomorphic seed, attempt to
   expand by one neuron at a time while preserving isomorphism.
7. Report the largest verified isomorphic circuit found across any triple of
   datasets, and write the submission CSV.

Key facts from the Codex FAQ that affect this script
-----------------------------------------------------
- FAFB and MCNS filter edges at >= 5 synapses.
- MANC and MAOL filter edges at >= 1 synapse.
- BANC filters at >= 3 synapses.
- The provided edge lists already reflect these thresholds — we use them as-is.
- Node IDs are dataset-specific and NOT comparable across datasets.
  Matching must be done via cell-type labels, not ID equality.

Dependencies
------------
    pip install pandas networkx requests tqdm
"""

import gzip
import io
import json
import urllib.request
from collections import defaultdict
from itertools import combinations, permutations
from pathlib import Path

import networkx as nx
import pandas as pd
import requests
from tqdm import tqdm

# ============================================================
# CONFIG
# ============================================================

OUTDIR = Path("results_v2")
OUTDIR.mkdir(exist_ok=True)

EDGE_LISTS = {
    "FAFB": "https://storage.googleapis.com/flywire-data/internship_projects/edge_lists/fafb_783_edge_list.csv",
    "BANC": "https://storage.googleapis.com/flywire-data/internship_projects/edge_lists/banc_626_edge_list.csv",
    "MANC": "https://storage.googleapis.com/flywire-data/internship_projects/edge_lists/manc_1.2.1_edge_list.csv",
    "MAOL": "https://storage.googleapis.com/flywire-data/internship_projects/edge_lists/maol_1.1_edge_list.csv",
    "MCNS": "https://storage.googleapis.com/flywire-data/internship_projects/edge_lists/mcns_0.9_edge_list.csv",
}

# Codex bulk-download API (only works for FAFB and BANC without login)
CODEX_DOWNLOAD = "https://codex.flywire.ai/api/download_resource"

# Known circuit seeds to try — cell type names as they appear in FlyWire annotations.
# These are canonical, well-studied Drosophila circuits with stereotyped wiring.
# Sources: Schlegel et al. 2024 (Nature), Dorkenwald et al. 2024 (Nature),
#          Costa et al. 2016 (Neuron - Giant Fiber), Reyn et al. 2014 (Neuron).
CIRCUIT_SEEDS = [
    # Giant Fiber escape circuit — the most stereotyped ~6-neuron circuit in fly
    ["GF", "PSI", "TTMn", "DLMn", "TPDO", "TDT"],

    # Central complex — navigation and sleep
    ["PEN_a", "PEN_b", "EPG", "PEG", "EL", "P-FN"],

    # Mushroom body output neurons (canonical lobes)
    ["MBON01", "MBON02", "MBON03", "MBON04", "MBON05", "MBON06"],

    # Optic lobe — motion detection T4/T5 (likely in MAOL/FAFB/MCNS)
    ["T4a", "T4b", "T4c", "T4d", "T5a", "T5b", "T5c", "T5d"],

    # Lobula plate tangential cells — optic flow
    ["LPLC1", "LPLC2", "LC4", "LC6", "LC10", "LC11"],
]

SRC = "source neuron id"
DST = "target neuron id"


# ============================================================
# STEP 1: Load edge lists
# ============================================================

def load_graphs():
    graphs = {}
    for name, url in EDGE_LISTS.items():
        print(f"Loading {name}...")
        df = pd.read_csv(url)
        G = nx.from_pandas_edgelist(
            df, source=SRC, target=DST,
            create_using=nx.DiGraph()
        )
        graphs[name] = G
        print(f"  {name}: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
    return graphs


# ============================================================
# STEP 2: Fetch cell-type metadata
# ============================================================

def fetch_codex_metadata(dataset: str) -> pd.DataFrame:
    """
    Fetch consolidated_cell_types from Codex for FAFB or BANC.
    Returns DataFrame with columns: root_id, cell_type (and others).
    Falls back gracefully if the resource is unavailable.
    """
    url = f"{CODEX_DOWNLOAD}?data_product=consolidated_cell_types&dataset={dataset.lower()}"
    print(f"  Fetching Codex metadata for {dataset} from {url}...")
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=60) as resp:
            raw = resp.read()
        # The response is gzip-compressed CSV
        with gzip.open(io.BytesIO(raw)) as f:
            df = pd.read_csv(f, low_memory=False)
        print(f"    Got {len(df):,} rows, columns: {list(df.columns)}")
        return df
    except Exception as e:
        print(f"    WARNING: Could not fetch {dataset} metadata: {e}")
        return pd.DataFrame()


def fetch_neuprint_metadata(dataset: str) -> pd.DataFrame:
    """
    Fetch neuron type annotations from Janelia neuPrint for MANC/MCNS.
    neuPrint has a public REST API that does not require authentication for
    publicly released datasets.

    neuPrint endpoint:
      https://neuprint.janelia.org/api/custom/custom
    with a Cypher query against the dataset server.

    Dataset server names:
      MANC  -> manc:v1.2.1
      MCNS  -> male-cns:v0.9  (also known as manc-vnc in some versions)
    """

    # neuPrint server map
    server_map = {
        "MANC": "manc:v1.2.1",
        "MCNS": "male-cns:v0.9",
    }

    if dataset not in server_map:
        print(f"  No neuPrint server configured for {dataset}, skipping.")
        return pd.DataFrame()

    server = server_map[dataset]
    endpoint = "https://neuprint.janelia.org/api/custom/custom"

    cypher = (
        "MATCH (n:Neuron) "
        "RETURN n.bodyId AS root_id, n.type AS cell_type, "
        "n.instance AS instance, n.status AS status "
        "LIMIT 200000"
    )

    payload = {"cypher": cypher, "dataset": server}

    print(f"  Fetching neuPrint metadata for {dataset} ({server})...")
    try:
        resp = requests.post(
            endpoint,
            json=payload,
            headers={"Content-Type": "application/json"},
            timeout=120,
        )
        resp.raise_for_status()
        data = resp.json()
        df = pd.DataFrame(data["data"], columns=data["columns"])
        df["root_id"] = df["root_id"].astype(str)
        print(f"    Got {len(df):,} rows")
        return df
    except Exception as e:
        print(f"    WARNING: neuPrint query failed for {dataset}: {e}")
        return pd.DataFrame()


def fetch_all_metadata() -> dict:
    """
    Returns dict: dataset_name -> DataFrame with at minimum columns
    [root_id (str), cell_type (str)].
    """
    meta = {}

    # FAFB and BANC via Codex
    for ds in ["FAFB", "BANC"]:
        df = fetch_codex_metadata(ds)
        if not df.empty:
            # Normalise column names
            df.columns = [c.lower().replace(" ", "_") for c in df.columns]
            if "root_id" not in df.columns:
                # Try alternative names
                for alt in ["id", "neuron_id", "cell_id"]:
                    if alt in df.columns:
                        df = df.rename(columns={alt: "root_id"})
                        break
            df["root_id"] = df["root_id"].astype(str)
            meta[ds] = df
        else:
            meta[ds] = pd.DataFrame()

    # MANC and MCNS via neuPrint
    for ds in ["MANC", "MCNS"]:
        df = fetch_neuprint_metadata(ds)
        if not df.empty:
            meta[ds] = df
        else:
            meta[ds] = pd.DataFrame()

    # MAOL — no easy public REST API; skip for now
    # (MAOL is optic-lobe only and the most specialised dataset)
    meta["MAOL"] = pd.DataFrame()

    return meta


# ============================================================
# STEP 3: Build cell-type → neuron-ID index per dataset
# ============================================================

def build_type_index(meta: dict, graphs: dict) -> dict:
    """
    Returns: { dataset: { cell_type_str: [node_id, ...] } }
    Only keeps neurons that actually appear in the graph.
    """
    index = {}
    for ds, df in meta.items():
        if df.empty or "cell_type" not in df.columns:
            index[ds] = {}
            continue

        G = graphs.get(ds)
        if G is None:
            index[ds] = {}
            continue

        graph_nodes = set(str(n) for n in G.nodes())

        type_map = defaultdict(list)
        for _, row in df.iterrows():
            ct = str(row.get("cell_type", "")).strip()
            rid = str(row.get("root_id", "")).strip()
            if ct and ct not in ("nan", "None", "") and rid in graph_nodes:
                type_map[ct].append(rid)

        index[ds] = dict(type_map)
        print(f"  {ds}: {len(type_map):,} cell types mapped to graph nodes")

    return index


# ============================================================
# STEP 4: Find cell types shared across >= 3 datasets
# ============================================================

def find_shared_types(type_index: dict, min_datasets: int = 3) -> list:
    """
    Returns list of (cell_type, {dataset: [node_ids]}) tuples
    for cell types present in >= min_datasets datasets.
    Sorted by total neuron count descending.
    """
    all_types = set()
    for ds_map in type_index.values():
        all_types.update(ds_map.keys())

    shared = []
    for ct in all_types:
        present = {
            ds: ids
            for ds, ids in type_index.items()
            if ct in type_index.get(ds, {})
        }
        if len(present) >= min_datasets:
            total = sum(len(v) for v in present.values())
            shared.append((ct, present, total))

    shared.sort(key=lambda x: x[2], reverse=True)
    print(f"\nFound {len(shared)} cell types present in >= {min_datasets} datasets")
    return shared


# ============================================================
# STEP 5: Exact isomorphism check for a given node correspondence
# ============================================================

def check_isomorphic(
    graphs: dict,
    correspondence: dict,
) -> bool:
    """
    Given a correspondence dict mapping dataset -> [node_ids],
    checks whether the induced directed subgraphs are mutually isomorphic
    under the ordering implied by the list positions.

    correspondence = {
        "FAFB": [id_A1, id_A2, id_A3],
        "MANC": [id_B1, id_B2, id_B3],
        "MCNS": [id_C1, id_C2, id_C3],
    }

    Isomorphism condition: for every pair (i, j),
      edge exists (list[i] -> list[j]) in dataset A
      iff it exists in dataset B and dataset C.

    This is O(N^2) in the number of matched nodes — fine for small circuits.
    """
    datasets = list(correspondence.keys())
    n = len(correspondence[datasets[0]])

    # Verify all lists are same length
    for ds in datasets:
        if len(correspondence[ds]) != n:
            return False

    # Build edge presence matrices
    edge_matrices = {}
    for ds in datasets:
        nodes = correspondence[ds]
        node_set = set(nodes)
        G = graphs[ds]
        mat = set()
        for i, u in enumerate(nodes):
            for j, v in enumerate(nodes):
                if i != j:
                    # Convert to same type as graph nodes
                    u_key = type(list(G.nodes())[0])(u) if G.number_of_nodes() > 0 else u
                    v_key = type(list(G.nodes())[0])(v) if G.number_of_nodes() > 0 else v
                    if G.has_edge(u_key, v_key):
                        mat.add((i, j))
        edge_matrices[ds] = mat

    # All matrices must be identical
    ref = edge_matrices[datasets[0]]
    for ds in datasets[1:]:
        if edge_matrices[ds] != ref:
            return False

    return True


def get_edge_matrix(G, nodes):
    """
    Returns set of (i, j) index pairs where edge nodes[i] -> nodes[j] exists.
    Handles both int and str node types.
    """
    if G.number_of_nodes() == 0:
        return set()

    # Determine node type in graph
    sample = next(iter(G.nodes()))
    def cast(x):
        try:
            return type(sample)(x)
        except Exception:
            return x

    mat = set()
    for i, u in enumerate(nodes):
        for j, v in enumerate(nodes):
            if i != j and G.has_edge(cast(u), cast(v)):
                mat.add((i, j))
    return mat


# ============================================================
# STEP 6: Try all orderings of small candidate groups
# ============================================================

def find_isomorphic_assignment(
    graphs: dict,
    candidate_groups: dict,
    max_neurons: int = 12,
) -> tuple:
    """
    For a cell type present in multiple datasets with one neuron per dataset
    per type (the ideal case), check if the induced subgraphs are isomorphic.

    For cases with multiple neurons per cell type per dataset, try all
    permutations of assignment (only feasible for small N).

    Returns (best_correspondence, n_nodes) or (None, 0).
    """
    datasets = list(candidate_groups.keys())
    if len(datasets) < 3:
        return None, 0

    # Take first 3 datasets with fewest neurons (most constrained)
    datasets_sorted = sorted(datasets, key=lambda d: len(candidate_groups[d]))
    trio = datasets_sorted[:3]

    lists = [candidate_groups[d] for d in trio]
    sizes = [len(l) for l in lists]

    # Skip if any list is empty
    if any(s == 0 for s in sizes):
        return None, 0

    # For efficiency, only try if sizes match (1:1 correspondence)
    # or sizes are small enough to permute
    if sizes[0] != sizes[1] or sizes[0] != sizes[2]:
        # Try with min size
        min_size = min(sizes)
        if min_size > max_neurons:
            return None, 0
        lists = [l[:min_size] for l in lists]
        sizes = [min_size] * 3

    if sizes[0] > max_neurons:
        return None, 0

    ref_nodes = lists[0]
    ref_mat = get_edge_matrix(graphs[trio[0]], ref_nodes)

    # Try all permutations of lists[1] and lists[2]
    best = None
    best_n = 0

    for perm1 in permutations(lists[1]):
        mat1 = get_edge_matrix(graphs[trio[1]], list(perm1))
        if mat1 != ref_mat:
            continue
        for perm2 in permutations(lists[2]):
            mat2 = get_edge_matrix(graphs[trio[2]], list(perm2))
            if mat2 == ref_mat:
                n = len(ref_nodes)
                if n > best_n:
                    best_n = n
                    best = {
                        trio[0]: list(ref_nodes),
                        trio[1]: list(perm1),
                        trio[2]: list(perm2),
                    }

    return best, best_n


# ============================================================
# STEP 7: Greedy circuit expansion
# ============================================================

def expand_circuit(
    graphs: dict,
    current: dict,
    type_index: dict,
    cell_type_lookup: dict,
    max_expand: int = 50,
) -> dict:
    """
    Given a verified isomorphic circuit (current), attempt to add one neuron
    at a time from the same datasets.

    Strategy: look at all neighbors of current circuit nodes in each dataset.
    For each neighbor, find its cell type. If that cell type exists in all
    three datasets with exactly one neuron each, try adding it.
    """
    datasets = list(current.keys())
    improved = True
    iterations = 0

    while improved and iterations < max_expand:
        improved = False
        iterations += 1

        # Collect candidate cell types from neighbors
        candidate_types = set()
        for ds in datasets:
            G = graphs[ds]
            current_nodes = set(current[ds])
            sample = next(iter(G.nodes())) if G.number_of_nodes() > 0 else None
            def cast(x, s=sample):
                try: return type(s)(x)
                except: return x

            for node in current_nodes:
                node_cast = cast(node)
                for nbr in list(G.predecessors(node_cast)) + list(G.successors(node_cast)):
                    nbr_str = str(nbr)
                    if nbr_str not in current_nodes:
                        ct = cell_type_lookup.get(ds, {}).get(nbr_str)
                        if ct:
                            candidate_types.add(ct)

        # Try each candidate type
        for ct in candidate_types:
            # Check it exists in all 3 datasets with exactly 1 neuron
            new_nodes = {}
            valid = True
            for ds in datasets:
                nodes = type_index.get(ds, {}).get(ct, [])
                # Exclude already-used neurons
                available = [n for n in nodes if n not in current[ds]]
                if len(available) != 1:
                    valid = False
                    break
                new_nodes[ds] = available[0]

            if not valid:
                continue

            # Try adding this neuron to the circuit
            trial = {ds: current[ds] + [new_nodes[ds]] for ds in datasets}

            # Check isomorphism of extended circuit
            mats = {}
            for ds in datasets:
                mats[ds] = get_edge_matrix(graphs[ds], trial[ds])

            ref = mats[datasets[0]]
            if all(mats[ds] == ref for ds in datasets[1:]):
                current = trial
                improved = True
                print(f"    Expanded circuit to {len(current[datasets[0]])} neurons "
                      f"by adding cell type '{ct}'")
                break

    return current


# ============================================================
# STEP 8: Seed-based fallback using known circuit names
# ============================================================

def try_known_seeds(
    graphs: dict,
    type_index: dict,
    cell_type_lookup: dict,
) -> tuple:
    """
    Try the known circuit seeds defined in CIRCUIT_SEEDS.
    For each seed, look up which datasets have ALL of the listed cell types,
    then attempt exact isomorphism.
    """
    best_circuit = None
    best_n = 0
    best_name = ""

    for seed_types in CIRCUIT_SEEDS:
        seed_name = "+".join(seed_types[:3]) + "..."
        print(f"\nTrying seed: {seed_name}")

        # Find datasets that have all types in this seed
        ds_coverage = {}
        for ds in graphs.keys():
            ti = type_index.get(ds, {})
            found = {}
            for ct in seed_types:
                # Exact match first, then prefix match
                if ct in ti:
                    found[ct] = ti[ct]
                else:
                    # Try case-insensitive prefix
                    matches = [k for k in ti.keys()
                               if k.lower().startswith(ct.lower())]
                    if matches:
                        found[ct] = ti[matches[0]]
            if len(found) == len(seed_types):
                ds_coverage[ds] = found

        if len(ds_coverage) < 3:
            print(f"  Only found in {len(ds_coverage)} datasets, need 3. Skipping.")
            continue

        print(f"  Found in datasets: {list(ds_coverage.keys())}")

        # Build correspondence: for each cell type, take first neuron from each ds
        # (assumes 1:1 for known circuits)
        for trio in combinations(list(ds_coverage.keys()), 3):
            correspondence = {}
            valid = True
            for ds in trio:
                nodes = []
                for ct in seed_types:
                    ct_nodes = ds_coverage[ds].get(ct, [])
                    if not ct_nodes:
                        valid = False
                        break
                    nodes.append(ct_nodes[0])
                if not valid:
                    break
                correspondence[ds] = nodes

            if not valid:
                continue

            # Check isomorphism
            mats = {ds: get_edge_matrix(graphs[ds], correspondence[ds])
                    for ds in trio}
            ref = mats[trio[0]]

            if all(mats[ds] == ref for ds in trio[1:]):
                n = len(correspondence[trio[0]])
                print(f"  ISOMORPHIC MATCH: {seed_name} across {trio}, "
                      f"N={n}, edges={len(ref)}")

                # Try to expand
                expanded = expand_circuit(
                    graphs, correspondence, type_index, cell_type_lookup
                )
                n_exp = len(expanded[list(expanded.keys())[0]])

                if n_exp > best_n:
                    best_n = n_exp
                    best_circuit = expanded
                    best_name = seed_name
            else:
                print(f"  Not isomorphic across {trio}")

    return best_circuit, best_n, best_name


# ============================================================
# STEP 9: Systematic search over all shared cell types
# ============================================================

def systematic_search(
    graphs: dict,
    shared_types: list,
    type_index: dict,
    cell_type_lookup: dict,
    top_k: int = 500,
) -> tuple:
    """
    For the top_k most common shared cell types, attempt to find
    isomorphic induced subgraphs.

    This is the brute-force complement to the seed-based approach.
    """
    best_circuit = None
    best_n = 0

    print(f"\nSystematic search over top {top_k} shared cell types...")

    for ct, present, total in tqdm(shared_types[:top_k]):
        datasets = list(present.keys())
        if len(datasets) < 3:
            continue

        # Only try if each dataset has exactly 1 neuron of this type
        # (avoids combinatorial explosion)
        sizes = {ds: len(present[ds]) for ds in datasets}
        if any(s != 1 for s in sizes.values()):
            continue

        # This is a single-neuron cell type — always isomorphic by itself (trivially)
        # We need to find GROUPS of such types that form isomorphic subgraphs
        # So we accumulate singleton types and try them in combinations
        # (handled separately below)

    # Build list of "singleton" cell types (exactly 1 neuron per dataset)
    # and try growing circuits from pairs/triplets of them
    singleton_types = []
    for ct, present, total in shared_types[:top_k]:
        datasets_with_type = list(present.keys())
        if len(datasets_with_type) < 3:
            continue
        sizes = [len(present[ds]) for ds in datasets_with_type]
        if all(s == 1 for s in sizes):
            singleton_types.append((ct, present))

    print(f"  Found {len(singleton_types)} singleton cell types across >= 3 datasets")

    if not singleton_types:
        return None, 0

    # Try pairs of singleton types as seeds
    print(f"  Trying pairs as isomorphic seeds...")

    for i, (ct1, pres1) in enumerate(tqdm(singleton_types[:200])):
        for ct2, pres2 in singleton_types[i+1:min(i+50, len(singleton_types))]:
            # Find datasets where both types exist
            common_ds = set(pres1.keys()) & set(pres2.keys())
            if len(common_ds) < 3:
                continue

            trio = list(common_ds)[:3]
            correspondence = {
                ds: [pres1[ds][0], pres2[ds][0]]
                for ds in trio
            }

            mats = {ds: get_edge_matrix(graphs[ds], correspondence[ds])
                    for ds in trio}
            ref = mats[trio[0]]

            if not all(mats[ds] == ref for ds in trio[1:]):
                continue

            # Found a 2-neuron isomorphic seed — try to expand
            expanded = expand_circuit(
                graphs, correspondence, type_index, cell_type_lookup
            )
            n_exp = len(expanded[list(expanded.keys())[0]])

            if n_exp > best_n:
                best_n = n_exp
                best_circuit = expanded
                print(f"  New best: N={n_exp} starting from {ct1}+{ct2} in {trio}")

    return best_circuit, best_n


# ============================================================
# STEP 10: Write submission CSV
# ============================================================

def write_submission(circuit: dict, out_path: Path):
    """
    Write the submission CSV with 3 columns (one per dataset) and N rows
    (one per matched neuron).
    """
    datasets = list(circuit.keys())
    rows = []
    n = len(circuit[datasets[0]])
    for i in range(n):
        row = {ds: circuit[ds][i] for ds in datasets}
        rows.append(row)

    df = pd.DataFrame(rows, columns=datasets)
    df.to_csv(out_path, index=False)
    print(f"\nSubmission written to {out_path}")
    print(df.to_string())


# ============================================================
# MAIN
# ============================================================

def main():
    print("=" * 70)
    print("FlyWire Circuit Discovery")
    print("=" * 70)

    # Step 1: Load graphs
    print("\n[1/6] Loading edge lists...")
    graphs = load_graphs()

    # Step 2: Fetch metadata
    print("\n[2/6] Fetching cell-type metadata...")
    meta = fetch_all_metadata()

    # Save metadata to disk for inspection
    for ds, df in meta.items():
        if not df.empty:
            out = OUTDIR / f"{ds}_metadata.csv"
            df.to_csv(out, index=False)
            print(f"  Saved {ds} metadata: {out}")

    # Step 3: Build type index
    print("\n[3/6] Building cell-type index...")
    type_index = build_type_index(meta, graphs)

    # Build reverse lookup: dataset -> node_id -> cell_type
    cell_type_lookup = {}
    for ds, df in meta.items():
        if df.empty or "cell_type" not in df.columns:
            cell_type_lookup[ds] = {}
            continue
        lookup = {}
        for _, row in df.iterrows():
            ct = str(row.get("cell_type", "")).strip()
            rid = str(row.get("root_id", "")).strip()
            if ct and ct not in ("nan", "None", ""):
                lookup[rid] = ct
        cell_type_lookup[ds] = lookup

    # Step 4: Find shared cell types
    print("\n[4/6] Finding shared cell types...")
    shared_types = find_shared_types(type_index, min_datasets=3)

    # Save shared type summary
    rows = []
    for ct, present, total in shared_types[:200]:
        row = {"cell_type": ct, "total_neurons": total}
        for ds in graphs.keys():
            row[f"n_{ds}"] = len(present.get(ds, []))
        rows.append(row)
    pd.DataFrame(rows).to_csv(OUTDIR / "shared_cell_types.csv", index=False)
    print(f"  Saved shared_cell_types.csv")

    # Step 5: Seed-based search
    print("\n[5/6] Seed-based search using known circuits...")
    best_circuit, best_n, best_name = try_known_seeds(
        graphs, type_index, cell_type_lookup
    )

    # Step 6: Systematic search (complement to seed search)
    print("\n[6/6] Systematic search over shared cell types...")
    sys_circuit, sys_n = systematic_search(
        graphs, shared_types, type_index, cell_type_lookup
    )

    if sys_n > best_n:
        best_n = sys_n
        best_circuit = sys_circuit
        best_name = "systematic_search"

    # Report
    print("\n" + "=" * 70)
    print("RESULT")
    print("=" * 70)

    if best_circuit is None:
        print("\nNo isomorphic circuit found.")
        print("\nDiagnosis:")
        print("  - If type_index is mostly empty, the Codex/neuPrint APIs")
        print("    may require authentication or have changed endpoints.")
        print("  - In that case, download metadata manually from:")
        print("    FAFB: https://codex.flywire.ai/api/download (select FAFB)")
        print("    MANC: https://www.janelia.org/project-team/flyem/manc-connectome")
        print("    MCNS: https://www.janelia.org/project-team/flyem/male-cns-connectome")
        print("  - Then load the CSVs and re-run from Step 3 onward.")
        print("\nFallback: inspect shared_cell_types.csv and manually verify")
        print("  isomorphism for the top entries using the Codex web UI.")
    else:
        print(f"\nBest circuit: N={best_n} neurons")
        print(f"Source: {best_name}")
        print(f"Datasets: {list(best_circuit.keys())}")

        out_csv = OUTDIR / "submission.csv"
        write_submission(best_circuit, out_csv)

        # Also verify the result
        datasets = list(best_circuit.keys())
        mats = {ds: get_edge_matrix(graphs[ds], best_circuit[ds])
                for ds in datasets}
        ref = mats[datasets[0]]
        all_match = all(mats[ds] == ref for ds in datasets[1:])
        print(f"\nIsomorphism verified: {all_match}")
        print(f"Edge count in shared circuit: {len(ref)}")


# ============================================================
# MANUAL FALLBACK HELPER
# ============================================================

def manual_check(graphs, neuron_ids_per_dataset):
    """
    Helper for manual verification when you have a specific set of neuron IDs.

    Usage:
        manual_check(graphs, {
            "FAFB": ["720575940596125868", "720575940605825666"],
            "MANC": ["10000", "10002"],
            "MCNS": ["10001", "10091"],
        })
    """
    datasets = list(neuron_ids_per_dataset.keys())
    mats = {}
    for ds in datasets:
        nodes = neuron_ids_per_dataset[ds]
        mat = get_edge_matrix(graphs[ds], nodes)
        mats[ds] = mat
        print(f"{ds}: {len(nodes)} nodes, {len(mat)} internal edges")
        print(f"  Edge matrix: {sorted(mat)}")

    ref = mats[datasets[0]]
    print(f"\nAll isomorphic: {all(mats[ds] == ref for ds in datasets[1:])}")
    return mats


if __name__ == "__main__":
    main()

FlyWire Circuit Discovery

[1/6] Loading edge lists...
Loading FAFB...
  FAFB: 138,584 nodes, 3,732,460 edges
Loading BANC...
  BANC: 112,885 nodes, 2,676,592 edges
Loading MANC...
  MANC: 23,641 nodes, 5,305,638 edges
Loading MAOL...
  MAOL: 51,669 nodes, 6,484,936 edges
Loading MCNS...
  MCNS: 165,820 nodes, 6,239,112 edges

[2/6] Fetching cell-type metadata...
  Fetching Codex metadata for FAFB from https://codex.flywire.ai/api/download_resource?data_product=consolidated_cell_types&dataset=fafb...
    Got 138,327 rows, columns: ['root_id', 'primary_type', 'additional_type(s)']
  Fetching Codex metadata for BANC from https://codex.flywire.ai/api/download_resource?data_product=consolidated_cell_types&dataset=banc...
  Fetching neuPrint metadata for MANC (manc:v1.2.1)...
  Fetching neuPrint metadata for MCNS (male-cns:v0.9)...
  Saved FAFB metadata: results_v2/FAFB_metadata.csv

[3/6] Building cell-type index...

[4/6] Finding shared cell types...

Found 0 cell types present in >= 3 da

0it [00:00, ?it/s]

  Found 0 singleton cell types across >= 3 datasets

RESULT

No isomorphic circuit found.

Diagnosis:
  - If type_index is mostly empty, the Codex/neuPrint APIs
    may require authentication or have changed endpoints.
  - In that case, download metadata manually from:
    FAFB: https://codex.flywire.ai/api/download (select FAFB)
    MANC: https://www.janelia.org/project-team/flyem/manc-connectome
    MCNS: https://www.janelia.org/project-team/flyem/male-cns-connectome
  - Then load the CSVs and re-run from Step 3 onward.

Fallback: inspect shared_cell_types.csv and manually verify
  isomorphism for the top entries using the Codex web UI.


In [ ]:
import pandas as pd
import networkx as nx
import numpy as np

from pathlib import Path
from collections import defaultdict

DATASETS = {
    "BANC": "data/BANC.csv",
    "FAFB": "data/FAFB.csv",
    "MANC": "data/MANC.csv",
    "MAOL": "data/MAOL.csv",
    "MCNS": "data/MCNS.csv",
}

SRC = "source neuron id"
DST = "target neuron id"

results = []

# --------------------------------------------------
# motif counters
# --------------------------------------------------

def count_reciprocals(G):

    reciprocal_edges = 0

    for u, v in G.edges():

        if u < v and G.has_edge(v, u):
            reciprocal_edges += 1

    return reciprocal_edges


def count_feedforward_loops(G):

    count = 0

    succ = {
        n: set(G.successors(n))
        for n in G.nodes()
    }

    for a in G.nodes():

        nbrs = succ[a]

        for b in nbrs:

            common = nbrs & succ[b]

            count += len(common)

    return count


def count_directed_3cycles(G):

    count = 0

    succ = {
        n: set(G.successors(n))
        for n in G.nodes()
    }

    for a in G.nodes():

        for b in succ[a]:

            for c in succ[b]:

                if c != a and a in succ[c]:
                    count += 1

    # every cycle counted 3 times
    return count // 3


# --------------------------------------------------
# run
# --------------------------------------------------

for name, path in DATASETS.items():

    print("=" * 70)
    print(name)

    df = pd.read_csv(path)

    G = nx.from_pandas_edgelist(
        df,
        source=SRC,
        target=DST,
        create_using=nx.DiGraph()
    )

    n = G.number_of_nodes()
    m = G.number_of_edges()

    print(f"nodes={n:,}")
    print(f"edges={m:,}")

    reciprocal = count_reciprocals(G)

    print("reciprocal done")

    ffl = count_feedforward_loops(G)

    print("feedforward done")

    cycles3 = count_directed_3cycles(G)

    print("3-cycle done")

    results.append({
        "dataset": name,

        "nodes": n,
        "edges": m,

        "reciprocal_pairs": reciprocal,
        "feedforward_loops": ffl,
        "directed_3cycles": cycles3,

        "reciprocal_per_1k_nodes":
            reciprocal / n * 1000,

        "ffl_per_1k_nodes":
            ffl / n * 1000,

        "cycles3_per_1k_nodes":
            cycles3 / n * 1000,

        "reciprocal_per_million_edges":
            reciprocal / m * 1e6,

        "ffl_per_million_edges":
            ffl / m * 1e6,

        "cycles3_per_million_edges":
            cycles3 / m * 1e6,
    })

# --------------------------------------------------
# save
# --------------------------------------------------

motif_df = pd.DataFrame(results)

motif_df.to_csv(
    "motif_summary.csv",
    index=False
)

print()
print("=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

display(
    motif_df.sort_values(
        "ffl_per_million_edges",
        ascending=False
    )
)

BANC
nodes=112,885
edges=2,676,592
reciprocal done
feedforward done
3-cycle done
FAFB
nodes=138,584
edges=3,732,460
reciprocal done
feedforward done
3-cycle done
MANC
nodes=23,641
edges=5,305,638
reciprocal done
feedforward done
3-cycle done
MAOL
nodes=51,669
edges=6,484,936
reciprocal done
feedforward done
3-cycle done
MCNS
nodes=165,820
edges=6,239,112
reciprocal done
feedforward done
3-cycle done

FINAL SUMMARY


,dataset,nodes,edges,reciprocal_pairs,feedforward_loops,directed_3cycles,reciprocal_per_1k_nodes,ffl_per_1k_nodes,cycles3_per_1k_nodes,reciprocal_per_million_edges,ffl_per_million_edges,cycles3_per_million_edges
2,MANC,23641,5305638,806074,320343871,70405011,34096.442621,1.355035e+07,2.978089e+06,151927.817164,6.037801e+07,1.326985e+07
3,MAOL,51669,6484936,913256,146667879,31578314,17675.124349,2.838605e+06,6.111656e+05,140827.295751,2.261670e+07,4.869487e+06
4,MCNS,165820,6239112,488924,41124597,6634788,2948.522494,2.480075e+05,4.001199e+04,78364.356979,6.591418e+06,1.063419e+06
0,BANC,112885,2676592,191712,16386213,2841316,1698.294725,1.451585e+05,2.517000e+04,71625.410223,6.122044e+06,1.061542e+06
1,FAFB,138584,3732460,310090,20060617,3552727,2237.559891,1.447542e+05,2.563591e+04,83079.256040,5.374637e+06,9.518460e+05


In [ ]:
# ============================================================
# FAST SPARSE MOTIF ANALYSIS
# ============================================================

import os
import json
import time
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm import tqdm

import scipy.sparse as sp

# ============================================================
# CONFIG
# ============================================================

DATASETS = {
    "BANC": "data/BANC.csv",
    "FAFB": "data/FAFB.csv",
    "MANC": "data/MANC.csv",
    "MAOL": "data/MAOL.csv",
    "MCNS": "data/MCNS.csv",
}

RESULT_DIR = Path("motif_sparse_results")
RESULT_DIR.mkdir(exist_ok=True)

# ============================================================
# LOGGING
# ============================================================

def log(msg):

    ts = time.strftime("%Y-%m-%d %H:%M:%S")

    print(f"[{ts}] {msg}")

# ============================================================
# LOAD GRAPH
# ============================================================

def load_sparse_graph(csv_path):

    df = pd.read_csv(csv_path)

    src_col = df.columns[0]
    dst_col = df.columns[1]

    nodes = pd.unique(
        pd.concat([
            df[src_col],
            df[dst_col]
        ])
    )

    node_to_idx = {
        n:i
        for i,n in enumerate(nodes)
    }

    rows = df[src_col].map(node_to_idx).to_numpy()
    cols = df[dst_col].map(node_to_idx).to_numpy()

    n = len(nodes)

    A = sp.csr_matrix(
        (
            np.ones(len(df), dtype=np.uint8),
            (rows, cols)
        ),
        shape=(n, n)
    )

    A.sum_duplicates()
    A.data[:] = 1

    return A, n, len(df)

# ============================================================
# MOTIF COUNTS
# ============================================================

def count_reciprocals(A):

    reciprocal_matrix = A.multiply(A.T)

    return int(
        reciprocal_matrix.sum() // 2
    )

# ------------------------------------------------------------

def count_ffl(A):

    A2 = A @ A

    ffl = A2.multiply(A)

    return int(ffl.sum())

# ------------------------------------------------------------

def count_directed_3cycles(A):

    A2 = A @ A
    A3 = A2 @ A

    trace = A3.diagonal().sum()

    return int(trace // 3)

# ============================================================
# ANALYZE ONE DATASET
# ============================================================

def analyze_dataset(name, csv_path):

    start = time.time()

    log("=" * 70)
    log(f"PROCESSING {name}")

    A, n_nodes, n_edges = load_sparse_graph(csv_path)

    log(f"Nodes: {n_nodes:,}")
    log(f"Edges: {n_edges:,}")

    log("Counting reciprocal pairs...")
    reciprocal = count_reciprocals(A)

    log("Counting feed-forward loops...")
    ffl = count_ffl(A)

    log("Counting directed 3-cycles...")
    cycles3 = count_directed_3cycles(A)

    runtime = time.time() - start

    result = {

        "dataset": name,

        "nodes": int(n_nodes),
        "edges": int(n_edges),

        "reciprocal_pairs": reciprocal,
        "feedforward_loops": ffl,
        "directed_3cycles": cycles3,

        "reciprocal_per_1k_nodes":
            reciprocal / (n_nodes / 1000),

        "ffl_per_1k_nodes":
            ffl / (n_nodes / 1000),

        "cycles3_per_1k_nodes":
            cycles3 / (n_nodes / 1000),

        "reciprocal_per_million_edges":
            reciprocal / (n_edges / 1_000_000),

        "ffl_per_million_edges":
            ffl / (n_edges / 1_000_000),

        "cycles3_per_million_edges":
            cycles3 / (n_edges / 1_000_000),

        "runtime_seconds":
            runtime
    }

    with open(
        RESULT_DIR / f"{name}.json",
        "w"
    ) as f:

        json.dump(
            result,
            f,
            indent=2
        )

    log(
        f"Finished in {runtime:.2f}s"
    )

    return result

# ============================================================
# MAIN
# ============================================================

all_results = []

for name, csv_path in DATASETS.items():

    checkpoint = RESULT_DIR / f"{name}.json"

    if checkpoint.exists():

        log(
            f"Skipping {name} (checkpoint exists)"
        )

        with open(checkpoint) as f:

            result = json.load(f)

        all_results.append(result)

        continue

    result = analyze_dataset(
        name,
        csv_path
    )

    all_results.append(result)

# ============================================================
# SAVE SUMMARY
# ============================================================

summary = pd.DataFrame(all_results)

summary = summary.sort_values(
    "ffl_per_million_edges",
    ascending=False
)

summary.to_csv(
    RESULT_DIR / "motif_summary.csv",
    index=False
)

print("\n")
print("=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

display(summary)

[2026-06-07 16:03:59] Skipping BANC (checkpoint exists)
[2026-06-07 16:03:59] ======================================================================
[2026-06-07 16:03:59] PROCESSING FAFB
[2026-06-07 16:04:01] Nodes: 138,584
[2026-06-07 16:04:01] Edges: 3,732,460
[2026-06-07 16:04:01] Counting reciprocal pairs...
[2026-06-07 16:04:01] Counting feed-forward loops...
[2026-06-07 16:04:15] Counting directed 3-cycles...
